In [1]:
# 1. Importing Libraries

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
path = r'C:\Users\bhava\Documents\18.02.26_Instacart Basket Analysis'

# Importing Data

In [5]:
df_custs_seg = pd.read_pickle(os.path.join(path, '02 Data', 'Prepared Data', 'ords_prods_merge_custs_viz_cfo.pkl'))

# 5. Customer Profiling using segment labels 

In [6]:
df_custs_seg['age'] = pd.to_numeric(df_custs_seg['age'], errors='coerce')

In [7]:
df_custs_seg = df_custs_seg.loc[df_custs_seg['low_activity_flag'] == 0]

In [8]:
df_custs_seg.copy()

,order_id,user_id,order_number,order_day_of_week,order_hour_of_day,days_since_last_order,product_id,add_to_cart_order,reordered,product_name,...,state,age,date_joined,number_of_dependants,marital_status,income,_merge,region,state_abbrev,low_activity_flag
0,2539329,1,1,2,8,NaN,196,1,0,Soda,...,Alabama,31,2/17/2019,3,married,40423,both,South,AL,0
1,2539329,1,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,...,Alabama,31,2/17/2019,3,married,40423,both,South,AL,0
2,2539329,1,1,2,8,NaN,12427,3,0,Original Beef Jerky,...,Alabama,31,2/17/2019,3,married,40423,both,South,AL,0
3,2539329,1,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,...,Alabama,31,2/17/2019,3,married,40423,both,South,AL,0
4,2539329,1,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,...,Alabama,31,2/17/2019,3,married,40423,both,South,AL,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32404854,2977660,206209,13,1,12,7.0,14197,5,1,Tomato Paste,...,Iowa,74,9/14/2019,3,married,137969,both,Midwest,IA,0
32404855,2977660,206209,13,1,12,7.0,38730,6,0,Brownie Crunch High Protein Bar,...,Iowa,74,9/14/2019,3,married,137969,both,Midwest,IA,0
32404856,2977660,206209,13,1,12,7.0,31477,7,0,High Protein Bar Chunky Peanut Butter,...,Iowa,74,9/14/2019,3,married,137969,both,Midwest,IA,0
32404857,2977660,206209,13,1,12,7.0,6567,8,0,Chocolate Peanut Butter Protein Bar,...,Iowa,74,9/14/2019,3,married,137969,both,Midwest,IA,0


In [10]:
df_custs_seg['age'].dtype

dtype('int64')

In [11]:
df_custs_seg['age'].isna().sum()

np.int64(0)

In [12]:
df_custs_seg['age'].describe()

count    3.096456e+07
mean     4.946803e+01
std      1.848528e+01
min      1.800000e+01
25%      3.300000e+01
50%      4.900000e+01
75%      6.500000e+01
max      8.100000e+01
Name: age, dtype: float64

In [13]:
df_custs_seg.shape

(30964564, 34)

In [23]:
df_custs_seg.columns

Index(['order_id', 'user_id', 'order_number', 'order_day_of_week',
       'order_hour_of_day', 'days_since_last_order', 'product_id',
       'add_to_cart_order', 'reordered', 'product_name', 'aisle_id',
       'department_id', 'prices', 'price_range_loc', 'busiest_day',
       'busiest_days', 'busiest_period_of_day', 'max_order', 'loyalty_flag',
       'avg_spend_user', 'spending_flag', 'ord_regularity_median',
       'frequency_flag', 'gender', 'state', 'age', 'date_joined',
       'number_of_dependants', 'marital_status', 'income', '_merge', 'region',
       'state_abbrev', 'low_activity_flag', 'age_group'],
      dtype='object')

## Creating Age Group Bins 

In [15]:
age_bins = [17, 29, 44, 100]

In [16]:
age_labels = ['Young Adult', 'Middle-Aged Adult', 'Older Adult']

In [17]:
df_custs_seg['age_group'] = pd.cut(df_custs_seg['age'], bins=age_bins, labels=age_labels, include_lowest=True)

In [19]:
df_custs_seg['age_group'].value_counts()

age_group
Older Adult          17885595
Middle-Aged Adult     7261366
Young Adult           5817603
Name: count, dtype: int64

#### Insights: The older adult segments makes up the highest number of customer in the dataset, followed by the Middle-Aged Adults.

## Creating family status segments

##### Customers with 1+ dependents are treated as "Family"
##### Customers with 0 dependents are treated as "Single"

In [24]:
df_custs_seg['family_status'] = np.where(df_custs_seg['number_of_dependants'] > 0, 'Family', 'Single')

In [28]:
df_custs_seg['family_status'].value_counts()

family_status
Family    23224883
Single     7739681
Name: count, dtype: int64

#### Uninque customers by family status (true customer counts)

In [29]:
df_custs_seg.groupby('family_status')['user_id'].nunique()

family_status
Family    121904
Single     40727
Name: user_id, dtype: int64

#### Insights: The customers are majorly families (121904) while the remaining have no dependants, i.e. single (40727) out of a total of 162631 unique customers. This means roughly 75% of customers are families. 

## Creating income segments 

In [30]:
income_bins = [0, 60000, 120000, df_custs_seg['income'].max()]

In [31]:
income_labels = ['Low Income', 'Middle Income', 'High Income']

In [32]:
df_custs_seg['income_group'] = pd.cut(df_custs_seg['income'], bins=income_bins, labels=income_labels, include_lowest=True)

In [33]:
df_custs_seg['income_group'].value_counts(dropna=False)

income_group
Middle Income    15981750
High Income       9179513
Low Income        5803301
Name: count, dtype: int64

#### Insights: Middle-income segment contains the highest number of customers in the dataframe, followed by high-income groups.

## General insight: We can see a pattern emerging here for revenue generating in this dataframe. Families of middle-income groups who are older adults represent the highest number of customers in the dataframe that the CFO is looking into, i.e. the ones with less than 5 max orders. Targetting this segment for marketing purposes would/could prove to be fruitful, if done correctly. 

## Creating department-based shopper profile

In [35]:
departments = pd.read_csv(os.path.join(path, "02 Data", "Original Data", "departments.csv"))

In [36]:
if set(["department_id", "department"]).issubset(departments.columns):
    departments_clean = departments[["department_id", "department"]].drop_duplicates("department_id").copy()
else:
    dept_series = departments.drop(columns=["department_id"], errors="ignore").iloc[0]
    departments_clean = dept_series.reset_index()
    departments_clean.columns = ["department_id", "department"]
    departments_clean["department_id"] = departments_clean["department_id"].astype(int)

In [37]:
df_custs_seg = df_custs_seg.merge(departments_clean, on="department_id", how="left")

In [38]:
# Chekck

print("department in df_custs_seg:", "department" in df_custs_seg.columns)

department in df_custs_seg: True


In [39]:
print("missing department names:", df_custs_seg["department"].isna().sum())

missing department names: 0


In [40]:
if "df_custs_seg" not in globals():
    raise NameError("df_custs_seg not found. Re-run Step 4 where you create df_custs_seg.")

In [43]:
if "department" not in df_custs_seg.columns:
    if "department_y" in df_custs_seg.columns or "department_x" in df_custs_seg.columns:
        df_custs_seg["department"] = df_custs_seg.get("department_y").combine_first(df_custs_seg.get("department_x"))
        df_custs_seg = df_custs_seg.drop(columns=[c for c in ["department_x", "department_y"] if c in df_custs_seg.columns])
    else:
        raise KeyError("No 'department' column found.")

### Department buckets

In [44]:
health_depts      = ["produce", "dairy eggs"]
family_depts      = ["pantry", "frozen", "dairy eggs"]
convenience_depts = ["frozen", "deli", "bakery"]
snack_depts       = ["snacks", "beverages"]
household_depts   = ["household", "personal care", "babies"]

### Classification of type of shopper based on the department 

In [45]:
def assign_shopper_profile(dept):
    # Handle missing safely
    if pd.isna(dept):
        return "Unknown / Missing Dept"

    dept = str(dept).strip().lower()

    if dept in health_depts:
        return "Health-Focused Shopper"
    elif dept in family_depts:
        return "Family Stock-Up Shopper"
    elif dept in convenience_depts:
        return "Convenience Shopper"
    elif dept in snack_depts:
        return "Snack & Beverage Shopper"
    elif dept in household_depts:
        return "Household Essentials Shopper"
    else:
        return "Mixed / Other Shopper"

In [57]:
df_custs_seg['department']

0                 beverages
1                dairy eggs
2                    snacks
3                    snacks
4                 household
                 ...       
30964559    dry goods pasta
30964560             snacks
30964561             snacks
30964562             snacks
30964563             snacks
Name: department, Length: 30964564, dtype: object

In [65]:
if "department" not in df_custs_seg.columns:
    df_custs_seg["department"] = df_custs_seg.get("department_y").combine_first(df_custs_seg.get("department_x"))

In [66]:
df_custs_seg["shopper_profile"] = df_custs_seg["department"].apply(assign_shopper_profile)

In [67]:
df_custs_seg["shopper_profile"].value_counts(dropna=False)

shopper_profile
Health-Focused Shopper          14256455
Snack & Beverage Shopper         5338307
Family Stock-Up Shopper          3904436
Mixed / Other Shopper            3806149
Convenience Shopper              2124662
Household Essentials Shopper     1534555
Name: count, dtype: int64

In [70]:
df_custs_seg.columns

Index(['order_id', 'user_id', 'order_number', 'order_day_of_week',
       'order_hour_of_day', 'days_since_last_order', 'product_id',
       'add_to_cart_order', 'reordered', 'product_name', 'aisle_id',
       'department_id', 'prices', 'price_range_loc', 'busiest_day',
       'busiest_days', 'busiest_period_of_day', 'max_order', 'loyalty_flag',
       'avg_spend_user', 'spending_flag', 'ord_regularity_median',
       'frequency_flag', 'gender', 'state', 'age', 'date_joined',
       'number_of_dependants', 'marital_status', 'income', '_merge', 'region',
       'state_abbrev', 'low_activity_flag', 'age_group', 'family_status',
       'income_group', 'department', 'shopper_profile',
       'missing_department_flag'],
      dtype='object')

#### Exporting dataset

In [74]:
df_custs_seg.to_pickle(os.path.join(path, "02 Data", "Prepared Data", "df_custs_seg.pkl"))

### Customer Persona - Compounding Data - Age, Family, Income and Behaviour

In [77]:
df_custs_seg['compound_persona'] = (df_custs_seg['age_group'].astype(str) + " | " +
    df_custs_seg['family_status'].astype(str) + " | " +
    df_custs_seg['income_group'].astype(str) + " | " +
    df_custs_seg['shopper_profile'].astype(str))

In [78]:
df_custs_seg['compound_persona'].nunique()

108

In [79]:
df_custs_seg['compound_persona'].value_counts().head(10)

compound_persona
Older Adult | Family | High Income | Health-Focused Shopper            2839555
Older Adult | Family | Middle Income | Health-Focused Shopper          2732286
Middle-Aged Adult | Family | Middle Income | Health-Focused Shopper    1551410
Young Adult | Family | Middle Income | Health-Focused Shopper          1377661
Older Adult | Single | High Income | Health-Focused Shopper             966055
Older Adult | Family | High Income | Snack & Beverage Shopper           953930
Older Adult | Family | Middle Income | Snack & Beverage Shopper         914499
Older Adult | Single | Middle Income | Health-Focused Shopper           905641
Older Adult | Family | High Income | Family Stock-Up Shopper            783806
Older Adult | Family | High Income | Mixed / Other Shopper              774972
Name: count, dtype: int64

In [80]:
df_custs_seg.groupby('compound_persona')['user_id'].nunique().sort_values(ascending=False).head(15)

compound_persona
Older Adult | Family | High Income | Health-Focused Shopper              30008
Older Adult | Family | High Income | Snack & Beverage Shopper            29322
Older Adult | Family | High Income | Family Stock-Up Shopper             29255
Older Adult | Family | High Income | Mixed / Other Shopper               28895
Older Adult | Family | Middle Income | Health-Focused Shopper            28339
Older Adult | Family | Middle Income | Snack & Beverage Shopper          27702
Older Adult | Family | Middle Income | Family Stock-Up Shopper           27591
Older Adult | Family | Middle Income | Mixed / Other Shopper             27296
Older Adult | Family | High Income | Convenience Shopper                 26729
Older Adult | Family | Middle Income | Convenience Shopper               25314
Older Adult | Family | High Income | Household Essentials Shopper        23333
Older Adult | Family | Middle Income | Household Essentials Shopper      21939
Middle-Aged Adult | Family | Middle